In [1]:
import pandas as pd
import requests
import numpy as np

In [2]:
from binance.client import Client
client = Client()  # sin API key para datos públicos
velas = client.get_historical_klines("BTCUSDT", Client.KLINE_INTERVAL_1HOUR,
                                     "1 Jan, 2024", "31 Dec, 2025")

In [3]:
velas_df = pd.DataFrame(velas, columns=["fecha", "open", "high", "low", "close", "volume",
                                       "close_time", "quote_asset_volume", "number_of_trades",
                                       "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"])

In [4]:
velas_df["fecha"] = pd.to_datetime(velas_df["fecha"], unit="ms")
velas_df["close_time"] = pd.to_datetime(velas_df["close_time"], unit="ms")

In [5]:
cols = velas_df.select_dtypes(include=[str]).columns

for col in cols:
    velas_df[col] = pd.to_numeric(velas_df[col])

In [6]:
velas_df.to_parquet("velas_df.parquet", engine="pyarrow", index=False)

In [8]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_parquet("velas_df.parquet")
sub = df[(df["fecha"] >= "2025-02-01") & (df["fecha"] < "2025-03-15")]  # un tramo para que se lea

fig = go.Figure(go.Candlestick(
    x=sub["fecha"],
    open=sub["open"], high=sub["high"], low=sub["low"], close=sub["close"],
    name="BTCUSDT 1h",
))
fig.update_layout(title="BTCUSDT 1h", xaxis_rangeslider_visible=False, height=600)
fig.show()

In [11]:
class Bar:
    def __init__(self, open, high, low, close, volume):

        if open <= 0 or high <= 0 or low <= 0 or close <= 0 or volume < 0:
            raise ValueError("Los valores de open, high, low y close deben ser mayores que cero, y volume no puede ser negativo.")

        if high < low or high < open or high < close:
            raise ValueError("El valor de high debe ser mayor o igual a low, open y close.")

        if low > high or low > open or low > close:
            raise ValueError("El valor de low debe ser menor o igual a high, open y close.")
        
        self.open = open
        self.high = high
        self.low = low
        self.close = close
        self.volume = volume

    @property
    def is_bullish(self):
        return self.close > self.open

    @property
    def is_bearish(self):
        return self.close < self.open

    @property
    def is_doji(self):
        return self.close == self.open

    @property
    def body_size(self):
        return abs(self.close - self.open)

In [12]:
candel = Bar(open=100, high=110, low=90, close=105, volume=1000)

In [14]:
candel.is_bullish

True